# Notebook 14 — Dual Gold Standard Comparison

**Research question:** Does the apparent pipeline advantage in NER disappear or reduce
when LLM outputs are evaluated against a gold standard built from LLM agreement rather
than pipeline agreement?

## Design

### Two gold standards
| Gold Standard | Built from | Language | Articles |
|---|---|---|---|
| Pipeline Gold | Full spaCy + Stanza + Flair agreement | German | 162 (filtered) |
| LLM Gold DE | Full Scout 17B + Qwen3 32B + Llama 70B DE agreement | German | 162 |

### Systems evaluated against both gold standards
| System | Input | Output language | Vs Pipeline Gold | Vs LLM Gold DE | Vs LLM Gold EN |
|---|---|---|---|---|---|
| spaCy | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Stanza | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Flair | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Llama 8B DE | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Scout 17B DE | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Qwen3 32B DE | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Llama 70B DE | German | German | ✅ clean | ✅ clean | ⚠ lang gap |
| Llama 8B EN | English | English | ⚠ lang gap | ⚠ lang gap | ✅ clean |
| Scout 17B EN | English | English | ⚠ lang gap | ⚠ lang gap | ✅ clean |
| Qwen3 32B EN | English | English | ⚠ lang gap | ⚠ lang gap | ✅ clean |
| Llama 70B EN | English | English | ⚠ lang gap | ⚠ lang gap | ✅ clean |

### Article set
All evaluations use the same **162 articles** — intersection of all four LLM DE runs.
This ensures every F1 value is directly comparable across gold standards and systems.

### Key outputs
1. F1 / Precision / Recall table — all systems vs both gold standards
2. Gold standard overlap — how similar are the two gold standards?
3. Fleiss' κ for LLM agreement — justifies LLM gold construction
4. Per-label breakdown — which entity types drive the differences


In [ ]:
# ── CELL 1 : INSTALLATION ────────────────────────────────────────────────────
!pip install -q sklearn
!pip install -q 'numpy>=2.0'   # must be last


In [ ]:
# ── CELL 2 : IMPORTS & CONFIGURATION ─────────────────────────────────────────

import os, json, pickle
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

ROOT        = Path('/content/drive/MyDrive/thesis')
DATA_PROC   = ROOT / 'Project/Data/Processed'
FIGURES_DIR = ROOT / 'Project/Outputs/Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Label normalisation ───────────────────────────────────────────────────────
LABEL_NORM = {
    "PERSON":"PER","person":"PER",
    "LOCATION":"LOC","location":"LOC","GPE":"LOC","gpe":"LOC","FAC":"LOC",
    "ORGANIZATION":"ORG","organisation":"ORG","organization":"ORG",
    "MISCELLANEOUS":"MISC","miscellaneous":"MISC",
    "EVENT":"MISC","PRODUCT":"MISC","LANGUAGE":"MISC","NORP":"MISC",
    "WORK_OF_ART":"MISC","LAW":"MISC","DATE":"MISC","TIME":"MISC","POL":"MISC",
    "PER":"PER","LOC":"LOC","ORG":"ORG","MISC":"MISC",
}
VALID_LABELS = {"PER","LOC","ORG","MISC"}

def normalize_entities(entity_list) -> set:
    out = set()
    for e in (entity_list or []):
        if isinstance(e, dict):
            text  = str(e.get('text','')).strip().lower()
            label = LABEL_NORM.get(str(e.get('label','')), None)
        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            text  = str(e[0]).strip().lower()
            label = LABEL_NORM.get(str(e[1]), None)
        else:
            continue
        if text and label in VALID_LABELS:
            out.add((text, label))
    return out

# ── System metadata ───────────────────────────────────────────────────────────
PIPELINE_SYSTEMS = {
    'spaCy' : 'ner_spacy',
    'Stanza': 'ner_stanza',
    'Flair' : 'ner_flair',
}

# Top 3 LLMs that build LLM Gold (by scale)
LLM_GOLD_MODELS = [
    'meta-llama/llama-4-scout-17b-16e-instruct',
    'qwen/qwen3-32b',
    'llama-3.3-70b-versatile',
]

# All 4 LLMs evaluated against gold standards
ALL_LLM_MODELS = {
    'llama-3.1-8b-instant'                     : {'scale_B': 8,  'family': 'llama'},
    'meta-llama/llama-4-scout-17b-16e-instruct': {'scale_B': 17, 'family': 'llama4'},
    'qwen/qwen3-32b'                           : {'scale_B': 32, 'family': 'qwen3'},
    'llama-3.3-70b-versatile'                  : {'scale_B': 70, 'family': 'llama'},
}

# Checkpoint filename mapping
def nb13b_ckpt(model_id: str, condition: str) -> Path:
    safe = model_id.replace('/', '_').replace('.', '_').replace('-', '_')
    # Match actual filenames on disk
    candidates = list(DATA_PROC.glob(f'nb13b_*{condition}_8k_checkpoint.pkl'))
    # Match by model substring
    model_short = model_id.split('/')[-1]
    for c in candidates:
        if any(part in c.name for part in model_short.split('-')[:3]):
            return c
    # Fallback: construct expected name
    safe2 = model_id.replace('/', '_').replace('-', '_').replace('.', '_')
    return DATA_PROC / f'nb13b_{safe2}_{condition}_8k_checkpoint.pkl'

print("✓ Configuration loaded")
print(f"  LLM Gold built from: {LLM_GOLD_MODELS}")
print(f"  Systems evaluated  : {len(PIPELINE_SYSTEMS)} pipelines + {len(ALL_LLM_MODELS)} LLMs × 2 conditions")


In [ ]:
# ── CELL 3 : LOAD ALL DATA ────────────────────────────────────────────────────

# ── Pipeline results ──────────────────────────────────────────────────────────
print("Loading pipeline results...")
pipeline_df = pd.read_pickle(DATA_PROC / 'ner_pipeline_results.pkl')
print(f"  shape: {pipeline_df.shape}")

pipeline_raw = {}   # {system_name: {article_id: set_of_(text,label)}}
for name, col in PIPELINE_SYSTEMS.items():
    if col not in pipeline_df.columns:
        print(f"  ⚠ {name}: column '{col}' missing"); continue
    lookup = {}
    for _, row in pipeline_df.iterrows():
        raw = row[col]
        lookup[row['article_id']] = (
            normalize_entities(raw) if isinstance(raw, (list, tuple)) else set())
    pipeline_raw[name] = lookup
    print(f"  ✓ {name}: {len(lookup)} articles")

# ── LLM DE-input checkpoints ──────────────────────────────────────────────────
print("
Loading LLM DE-input checkpoints...")
llm_de_raw = {}   # {model_id: {article_id: set_of_(text,label)}}

# List actual nb13b checkpoint files for matching
all_ckpt_files = list(DATA_PROC.glob('nb13b_*_de_8k_checkpoint.pkl'))
print(f"  Found {len(all_ckpt_files)} DE checkpoint files:")
for f in sorted(all_ckpt_files):
    print(f"    {f.name}")

for model_id in ALL_LLM_MODELS:
    # Find matching checkpoint file
    model_short = model_id.split('/')[-1].lower()
    parts       = [p for p in model_short.replace('-','_').split('_') if len(p) > 2]
    matched     = None
    for f in all_ckpt_files:
        fname_lower = f.name.lower()
        if sum(1 for p in parts[:4] if p in fname_lower) >= 3:
            matched = f; break
    if matched is None:
        print(f"  ⚠ No checkpoint found for {model_id}"); continue
    with open(matched, 'rb') as f:
        ckpt = pickle.load(f)
    lookup = {r['article_id']: normalize_entities(r.get('entities',[]))
              for r in ckpt.get('results',[]) if r.get('status')=='ok'}
    llm_de_raw[model_id] = lookup
    print(f"  ✓ {model_id} DE: {len(lookup)} articles  ({matched.name})")

# ── LLM EN-input checkpoints ──────────────────────────────────────────────────
print("
Loading LLM EN-input checkpoints...")
llm_en_raw = {}

all_ckpt_en = list(DATA_PROC.glob('nb13b_*_en_8k_checkpoint.pkl'))
for model_id in ALL_LLM_MODELS:
    model_short = model_id.split('/')[-1].lower()
    parts       = [p for p in model_short.replace('-','_').split('_') if len(p) > 2]
    matched     = None
    for f in all_ckpt_en:
        fname_lower = f.name.lower()
        if sum(1 for p in parts[:4] if p in fname_lower) >= 3:
            matched = f; break
    if matched is None:
        print(f"  ⚠ No checkpoint found for {model_id}"); continue
    with open(matched, 'rb') as f:
        ckpt = pickle.load(f)
    lookup = {r['article_id']: normalize_entities(r.get('entities',[]))
              for r in ckpt.get('results',[]) if r.get('status')=='ok'}
    llm_en_raw[model_id] = lookup
    print(f"  ✓ {model_id} EN: {len(lookup)} articles  ({matched.name})")


In [ ]:
# ── CELL 4 : IDENTIFY SHARED ARTICLE SET (162 articles) ──────────────────────
# Intersection of all four LLM DE runs — Llama 70B DE is binding constraint.
# ALL evaluations in this notebook use this same article set.

de_sets = [set(llm_de_raw[m].keys()) for m in LLM_GOLD_MODELS
           if m in llm_de_raw]

if len(de_sets) < len(LLM_GOLD_MODELS):
    missing = [m for m in LLM_GOLD_MODELS if m not in llm_de_raw]
    print(f"⚠ Missing DE checkpoints for: {missing}")
    print("  Cannot build LLM gold — check checkpoint files")
else:
    SHARED_IDS = sorted(set.intersection(*de_sets))
    print(f"Shared article set: {len(SHARED_IDS)} articles")
    print(f"  Scout 17B DE : {len(list(llm_de_raw.items())[1][1])} articles")
    print(f"  Qwen3 32B DE : {len(llm_de_raw.get('qwen/qwen3-32b',{}))} articles")
    print(f"  Llama 70B DE : {len(llm_de_raw.get('llama-3.3-70b-versatile',{}))} articles")
    print(f"  Intersection : {len(SHARED_IDS)} articles  ← used for ALL evaluations")

    # Also verify pipeline and EN-input runs cover this set
    for name, lookup in pipeline_raw.items():
        covered = sum(1 for aid in SHARED_IDS if aid in lookup)
        print(f"  Pipeline {name}: {covered}/{len(SHARED_IDS)} shared articles covered")
    for model_id, lookup in llm_en_raw.items():
        covered = sum(1 for aid in SHARED_IDS if aid in lookup)
        short   = model_id.split('/')[-1][:25]
        print(f"  LLM EN {short}: {covered}/{len(SHARED_IDS)} shared articles covered")

SHARED_IDS_SET = set(SHARED_IDS)


In [ ]:
# ── CELL 5 : BUILD GOLD STANDARDS ────────────────────────────────────────────

# ── Pipeline Gold (German) ────────────────────────────────────────────────────
# Full spaCy + Stanza + Flair agreement, filtered to SHARED_IDS
pipeline_gold = {}
total_pg_ents = 0
for aid in SHARED_IDS:
    sets = [pipeline_raw[name].get(aid, set())
            for name in PIPELINE_SYSTEMS if name in pipeline_raw]
    if len(sets) == 3:
        agreed = sets[0] & sets[1] & sets[2]   # full 3/3 agreement
    else:
        agreed = set()
    pipeline_gold[aid] = agreed
    total_pg_ents += len(agreed)

print(f"Pipeline Gold (German):")
print(f"  Articles : {len(pipeline_gold)}")
print(f"  Entities : {total_pg_ents}")
print(f"  Mean/art : {total_pg_ents/len(pipeline_gold):.1f}")

# Label distribution
pg_labels = defaultdict(int)
for ents in pipeline_gold.values():
    for _, label in ents:
        pg_labels[label] += 1
print(f"  Labels   : {dict(sorted(pg_labels.items()))}")

# ── LLM Gold DE (German) ─────────────────────────────────────────────────────
# Full Scout 17B + Qwen3 32B + Llama 70B DE agreement, on SHARED_IDS
llm_gold_de = {}
total_lg_ents = 0
for aid in SHARED_IDS:
    sets = [llm_de_raw[m].get(aid, set()) for m in LLM_GOLD_MODELS
            if m in llm_de_raw]
    if len(sets) == 3:
        agreed = sets[0] & sets[1] & sets[2]
    else:
        agreed = set()
    llm_gold_de[aid] = agreed
    total_lg_ents += len(agreed)

print(f"
LLM Gold DE (German):")
print(f"  Articles : {len(llm_gold_de)}")
print(f"  Entities : {total_lg_ents}")
print(f"  Mean/art : {total_lg_ents/len(llm_gold_de):.1f}")

lg_labels = defaultdict(int)
for ents in llm_gold_de.values():
    for _, label in ents:
        lg_labels[label] += 1
print(f"  Labels   : {dict(sorted(lg_labels.items()))}")

# ── LLM Gold EN (English) ────────────────────────────────────────────────────
# Full Scout 17B + Qwen3 32B + Llama 70B EN agreement, on SHARED_IDS
llm_gold_en = {}
total_le_ents = 0
for aid in SHARED_IDS:
    sets = [llm_en_raw[m].get(aid, set()) for m in LLM_GOLD_MODELS
            if m in llm_en_raw]
    if len(sets) == 3:
        agreed = sets[0] & sets[1] & sets[2]
    else:
        agreed = set()
    llm_gold_en[aid] = agreed
    total_le_ents += len(agreed)

print(f"
LLM Gold EN (English):")
print(f"  Articles : {len(llm_gold_en)}")
print(f"  Entities : {total_le_ents}")
print(f"  Mean/art : {total_le_ents/len(llm_gold_en):.1f}")

le_labels = defaultdict(int)
for ents in llm_gold_en.values():
    for _, label in ents:
        le_labels[label] += 1
print(f"  Labels   : {dict(sorted(le_labels.items()))}")


In [ ]:
# ── CELL 6 : FLEISS' κ FOR LLM AGREEMENT + GOLD STANDARD OVERLAP ─────────────

from itertools import combinations

def fleiss_kappa_entities(model_lookups: dict, article_ids: list) -> float:
    """
    Compute Fleiss' κ across multiple entity extraction systems.
    Uses entity presence/absence as binary ratings per unique entity span.
    """
    n_raters = len(model_lookups)
    lookups  = list(model_lookups.values())

    # Collect all unique entities across all articles and models
    all_entities_per_art = {}
    for aid in article_ids:
        universe = set()
        for lookup in lookups:
            universe |= lookup.get(aid, set())
        all_entities_per_art[aid] = universe

    # Build rating matrix: rows = entity instances, cols = categories (found/not)
    total_items = sum(len(v) for v in all_entities_per_art.values())
    if total_items == 0:
        return 0.0

    # P_i: proportion of agreeing pairs for each entity
    P_i_sum = 0.0
    category_counts = defaultdict(int)  # how many times each category assigned

    for aid, universe in all_entities_per_art.items():
        for ent in universe:
            n_found = sum(1 for lookup in lookups if ent in lookup.get(aid, set()))
            n_not   = n_raters - n_found
            # P_i for this entity
            if n_raters > 1:
                P_i = (n_found*(n_found-1) + n_not*(n_not-1)) / (n_raters*(n_raters-1))
            else:
                P_i = 1.0
            P_i_sum += P_i
            category_counts['found']     += n_found
            category_counts['not_found'] += n_not

    P_bar = P_i_sum / total_items

    # Expected agreement P_e
    total_ratings = total_items * n_raters
    p_found    = category_counts['found']     / total_ratings
    p_not      = category_counts['not_found'] / total_ratings
    P_e        = p_found**2 + p_not**2

    if P_e == 1.0:
        return 1.0

    kappa = (P_bar - P_e) / (1 - P_e)
    return round(float(kappa), 3)


# ── Compute κ for LLM DE agreement ───────────────────────────────────────────
print("Computing Fleiss' κ for LLM DE agreement (top 3 models)...")
gold_model_lookups = {m: llm_de_raw[m] for m in LLM_GOLD_MODELS if m in llm_de_raw}
kappa_llm_de = fleiss_kappa_entities(gold_model_lookups, SHARED_IDS)
print(f"  Fleiss' κ (LLM DE, top 3): {kappa_llm_de}")
print(f"  Pipeline κ for reference : 0.717 (from nb03)")

# Interpretation
if kappa_llm_de >= 0.6:
    interp = "substantial agreement — LLM gold construction justified"
elif kappa_llm_de >= 0.4:
    interp = "moderate agreement — LLM gold usable with caveat"
else:
    interp = "fair/poor agreement — LLM gold construction questionable"
print(f"  Interpretation: {interp}")

# ── Gold standard overlap ─────────────────────────────────────────────────────
# How similar are Pipeline Gold and LLM Gold DE?
print("
Gold standard overlap (Pipeline Gold vs LLM Gold DE):")
jaccard_scores, overlap_counts = [], []
for aid in SHARED_IDS:
    pg  = pipeline_gold.get(aid, set())
    lg  = llm_gold_de.get(aid, set())
    if pg or lg:
        j = len(pg & lg) / len(pg | lg) if (pg | lg) else 1.0
        jaccard_scores.append(j)
    overlap_counts.append(len(pg & lg))

print(f"  Mean Jaccard similarity  : {np.mean(jaccard_scores):.3f}")
print(f"  Median Jaccard           : {np.median(jaccard_scores):.3f}")
print(f"  Total entities in both   : {sum(overlap_counts)}")
print(f"  Pipeline Gold only       : "
      f"{sum(len(pipeline_gold[a]) for a in SHARED_IDS) - sum(overlap_counts)}")
print(f"  LLM Gold DE only         : "
      f"{sum(len(llm_gold_de[a]) for a in SHARED_IDS) - sum(overlap_counts)}")
print(f"  In both gold standards   : {sum(overlap_counts)}")

overlap_pct_pg = sum(overlap_counts) / sum(len(pipeline_gold[a]) for a in SHARED_IDS) * 100
overlap_pct_lg = sum(overlap_counts) / sum(len(llm_gold_de[a]) for a in SHARED_IDS) * 100
print(f"  % of Pipeline Gold covered by LLM Gold: {overlap_pct_pg:.1f}%")
print(f"  % of LLM Gold covered by Pipeline Gold: {overlap_pct_lg:.1f}%")


In [ ]:
# ── CELL 7 : COMPUTE F1 / PRECISION / RECALL AGAINST BOTH GOLD STANDARDS ─────

def compute_prf(predicted: set, gold: set) -> tuple:
    """Returns (precision, recall, f1) for entity sets."""
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*p*r / (p+r)  if (p + r)  > 0 else 0.0
    return round(p,4), round(r,4), round(f1,4)

def macro_prf(system_lookup: dict, gold_lookup: dict, article_ids: list) -> dict:
    """Macro-averaged P/R/F1 across articles. Also returns per-label F1."""
    p_scores, r_scores, f1_scores = [], [], []
    label_tp = defaultdict(int)
    label_fp = defaultdict(int)
    label_fn = defaultdict(int)

    for aid in article_ids:
        pred = system_lookup.get(aid, set())
        gold = gold_lookup.get(aid, set())
        p, r, f1 = compute_prf(pred, gold)
        p_scores.append(p); r_scores.append(r); f1_scores.append(f1)

        for ent in pred & gold:
            label_tp[ent[1]] += 1
        for ent in pred - gold:
            label_fp[ent[1]] += 1
        for ent in gold - pred:
            label_fn[ent[1]] += 1

    per_label = {}
    for lbl in VALID_LABELS:
        tp = label_tp[lbl]; fp = label_fp[lbl]; fn = label_fn[lbl]
        lp = tp/(tp+fp) if (tp+fp) > 0 else 0.0
        lr = tp/(tp+fn) if (tp+fn) > 0 else 0.0
        lf = 2*lp*lr/(lp+lr) if (lp+lr) > 0 else 0.0
        per_label[lbl] = round(lf, 4)

    return {
        'P'         : round(float(np.mean(p_scores)),  4),
        'R'         : round(float(np.mean(r_scores)),  4),
        'F1'        : round(float(np.mean(f1_scores)), 4),
        'per_label' : per_label,
    }

# ── Build all system lookups ──────────────────────────────────────────────────
# Pipelines (German)
system_lookups = {}
for name, lookup in pipeline_raw.items():
    system_lookups[f'{name} (pipeline)'] = lookup

# LLMs DE-input (German)
for model_id, lookup in llm_de_raw.items():
    short = model_id.split('/')[-1][:20]
    system_lookups[f'{short} (LLM DE)'] = lookup

# LLMs EN-input (English — language gap when vs German gold)
for model_id, lookup in llm_en_raw.items():
    short = model_id.split('/')[-1][:20]
    system_lookups[f'{short} (LLM EN)'] = lookup

# ── Evaluate against all three gold standards ─────────────────────────────────
GOLD_STANDARDS = {
    'Pipeline Gold (DE)': pipeline_gold,
    'LLM Gold DE'       : llm_gold_de,
    'LLM Gold EN'       : llm_gold_en,
}

results = {}   # {system_name: {gold_name: {P, R, F1, per_label}}}
for sys_name, sys_lookup in system_lookups.items():
    results[sys_name] = {}
    for gold_name, gold_lookup in GOLD_STANDARDS.items():
        results[sys_name][gold_name] = macro_prf(
            sys_lookup, gold_lookup, SHARED_IDS)

print("✓ F1 computation complete")
print(f"  {len(results)} systems × {len(GOLD_STANDARDS)} gold standards")


In [ ]:
# ── CELL 8 : RESULTS TABLES ───────────────────────────────────────────────────

# ── Table 1 : F1 comparison — all systems vs both DE gold standards ───────────
print("=" * 70)
print("TABLE 1 — F1 SCORES: Pipeline Gold vs LLM Gold DE")
print("(Language gap ⚠ = English entities vs German gold — lower bound)")
print("=" * 70)

rows = []
for sys_name in system_lookups:
    pg_f1 = results[sys_name]['Pipeline Gold (DE)']['F1']
    lg_f1 = results[sys_name]['LLM Gold DE']['F1']
    diff  = round(lg_f1 - pg_f1, 4)
    is_llm_en = '(LLM EN)' in sys_name
    note  = ' ⚠' if is_llm_en else ''
    rows.append({
        'System'          : sys_name + note,
        'Vs Pipeline Gold': pg_f1,
        'Vs LLM Gold DE'  : lg_f1,
        'Difference'      : f'{diff:+.4f}',
    })

t1 = pd.DataFrame(rows)
# Sort: pipelines first, then LLM DE, then LLM EN, within each by F1 desc
def sort_key(row):
    s = row['System']
    if 'pipeline' in s: return (0, -row['Vs Pipeline Gold'])
    if 'LLM DE'   in s: return (1, -row['Vs Pipeline Gold'])
    return (2, -row['Vs Pipeline Gold'])
t1 = t1.sort_values('System', key=lambda x: x.map(
    lambda s: (0 if 'pipeline' in s else 1 if 'LLM DE' in s else 2,
               -results[s.replace(' ⚠','')]['Pipeline Gold (DE)']['F1']
               if s.replace(' ⚠','') in results else 0)))
print(t1.to_string(index=False))

# ── Table 2 : Full P/R/F1 ────────────────────────────────────────────────────
print("
" + "=" * 70)
print("TABLE 2 — FULL P / R / F1 AGAINST PIPELINE GOLD (DE)")
print("=" * 70)
rows2 = []
for sys_name in system_lookups:
    r = results[sys_name]['Pipeline Gold (DE)']
    rows2.append({'System': sys_name,
                  'P': r['P'], 'R': r['R'], 'F1': r['F1']})
t2 = pd.DataFrame(rows2).sort_values('F1', ascending=False)
print(t2.to_string(index=False))

print("
" + "=" * 70)
print("TABLE 3 — FULL P / R / F1 AGAINST LLM GOLD DE")
print("=" * 70)
rows3 = []
for sys_name in system_lookups:
    r = results[sys_name]['LLM Gold DE']
    rows3.append({'System': sys_name,
                  'P': r['P'], 'R': r['R'], 'F1': r['F1']})
t3 = pd.DataFrame(rows3).sort_values('F1', ascending=False)
print(t3.to_string(index=False))

# ── Table 4 : LLM EN-input vs LLM Gold EN (clean comparison) ─────────────────
print("
" + "=" * 70)
print("TABLE 4 — LLM EN INPUT vs LLM GOLD EN (both English — no language gap)")
print("=" * 70)
rows4 = []
for sys_name in system_lookups:
    if '(LLM EN)' not in sys_name:
        continue
    r = results[sys_name]['LLM Gold EN']
    rows4.append({'System': sys_name,
                  'P': r['P'], 'R': r['R'], 'F1': r['F1']})
t4 = pd.DataFrame(rows4).sort_values('F1', ascending=False)
print(t4.to_string(index=False))

# ── Table 5 : Per-label F1 breakdown ─────────────────────────────────────────
print("
" + "=" * 70)
print("TABLE 5 — PER-LABEL F1 AGAINST PIPELINE GOLD (DE)")
print("=" * 70)
rows5 = []
for sys_name in system_lookups:
    pl = results[sys_name]['Pipeline Gold (DE)']['per_label']
    rows5.append({'System': sys_name, **pl,
                  'Macro F1': results[sys_name]['Pipeline Gold (DE)']['F1']})
t5 = pd.DataFrame(rows5).sort_values('Macro F1', ascending=False)
print(t5.to_string(index=False))


In [ ]:
# ── CELL 9 : PLOTS ────────────────────────────────────────────────────────────

# ── Figure 1 : F1 comparison — Pipeline Gold vs LLM Gold DE ──────────────────
fig, ax = plt.subplots(figsize=(12, 7))

sys_names  = list(system_lookups.keys())
pg_f1s     = [results[s]['Pipeline Gold (DE)']['F1'] for s in sys_names]
lg_f1s     = [results[s]['LLM Gold DE']['F1']        for s in sys_names]

x      = np.arange(len(sys_names))
width  = 0.38
colors = {'pipeline': ('#1f77b4','#aec7e8'),
          'LLM DE'  : ('#2ca02c','#98df8a'),
          'LLM EN'  : ('#ff7f0e','#ffbb78')}

bar_colors_pg, bar_colors_lg = [], []
for s in sys_names:
    if 'pipeline' in s:
        bar_colors_pg.append(colors['pipeline'][0])
        bar_colors_lg.append(colors['pipeline'][1])
    elif 'LLM DE' in s:
        bar_colors_pg.append(colors['LLM DE'][0])
        bar_colors_lg.append(colors['LLM DE'][1])
    else:
        bar_colors_pg.append(colors['LLM EN'][0])
        bar_colors_lg.append(colors['LLM EN'][1])

b1 = ax.bar(x - width/2, pg_f1s, width, label='Vs Pipeline Gold (DE)',
            color=bar_colors_pg, edgecolor='black', linewidth=0.5, alpha=0.9)
b2 = ax.bar(x + width/2, lg_f1s, width, label='Vs LLM Gold DE',
            color=bar_colors_lg, edgecolor='black', linewidth=0.5, alpha=0.9)

# Value labels
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=6.5, rotation=90)

# Short labels for x-axis
short_labels = []
for s in sys_names:
    if 'pipeline' in s:
        short_labels.append(s.replace(' (pipeline)',''))
    elif 'LLM DE' in s:
        short_labels.append(s.replace(' (LLM DE)','
(DE)'))
    else:
        short_labels.append(s.replace(' (LLM EN)','
(EN)⚠'))

ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=8, rotation=30, ha='right')
ax.set_ylabel('Macro F1', fontsize=11)
ax.set_title('F1 Scores: Pipeline Gold vs LLM Gold DE
'
             '(Blue=pipeline systems · Green=LLM DE-input · Orange=LLM EN-input ⚠)
'
             f'({len(SHARED_IDS)} articles · ⚠ = language gap — lower bound)',
             fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(FIGURES_DIR / f'nb14_f1_dual_gold.{ext}',
                dpi=300 if ext == 'pdf' else 150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: nb14_f1_dual_gold.pdf/.png")


# ── Figure 2 : Gold standard similarity ──────────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Entity count comparison
gold_names  = ['Pipeline Gold
(DE)', 'LLM Gold
(DE)', 'LLM Gold
(EN)']
gold_counts = [
    sum(len(pipeline_gold[a]) for a in SHARED_IDS),
    sum(len(llm_gold_de[a])   for a in SHARED_IDS),
    sum(len(llm_gold_en[a])   for a in SHARED_IDS),
]
axes2[0].bar(gold_names, gold_counts,
             color=['#1f77b4','#2ca02c','#ff7f0e'],
             edgecolor='black', linewidth=0.5, alpha=0.9)
for i, (name, count) in enumerate(zip(gold_names, gold_counts)):
    axes2[0].text(i, count + 20, str(count),
                  ha='center', va='bottom', fontsize=10, fontweight='bold')
axes2[0].set_ylabel('Total entities', fontsize=11)
axes2[0].set_title(f'Gold Standard Size
({len(SHARED_IDS)} articles)',
                   fontsize=11)
axes2[0].grid(axis='y', alpha=0.3)

# Panel B: Per-label distribution
labels_list = sorted(VALID_LABELS)
x2      = np.arange(len(labels_list))
width2  = 0.28
pg_cnts = [pg_labels.get(l, 0) for l in labels_list]
lg_cnts = [lg_labels.get(l, 0) for l in labels_list]
le_cnts = [le_labels.get(l, 0) for l in labels_list]

axes2[1].bar(x2 - width2, pg_cnts, width2, label='Pipeline Gold (DE)',
             color='#1f77b4', edgecolor='black', lw=0.5, alpha=0.9)
axes2[1].bar(x2,           lg_cnts, width2, label='LLM Gold DE',
             color='#2ca02c', edgecolor='black', lw=0.5, alpha=0.9)
axes2[1].bar(x2 + width2, le_cnts, width2, label='LLM Gold EN',
             color='#ff7f0e', edgecolor='black', lw=0.5, alpha=0.9)
axes2[1].set_xticks(x2)
axes2[1].set_xticklabels(labels_list, fontsize=11)
axes2[1].set_ylabel('Entity count', fontsize=11)
axes2[1].set_title('Label Distribution by Gold Standard', fontsize=11)
axes2[1].legend(fontsize=9)
axes2[1].grid(axis='y', alpha=0.3)

fig2.suptitle('Gold Standard Comparison', fontsize=13)
plt.tight_layout()
for ext in ['pdf', 'png']:
    fig2.savefig(FIGURES_DIR / f'nb14_gold_comparison.{ext}',
                 dpi=300 if ext == 'pdf' else 150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: nb14_gold_comparison.pdf/.png")


# ── Figure 3 : Pipeline vs LLM F1 under both gold standards ──────────────────
# Shows whether pipeline advantage shrinks under LLM gold
fig3, axes3 = plt.subplots(1, 2, figsize=(13, 6), sharey=True)

def system_short(s):
    s = s.replace(' (pipeline)','').replace(' (LLM DE)','
(DE)')
    s = s.replace(' (LLM EN)','
(EN)⚠')
    return s

for ax3, (gold_name, gold_col) in zip(axes3,
        [('Pipeline Gold (DE)', '#1f77b4'),
         ('LLM Gold DE',        '#2ca02c')]):
    f1s   = [(system_short(s), results[s][gold_name]['F1'])
             for s in sys_names]
    f1s.sort(key=lambda x: -x[1])
    names_sorted = [x[0] for x in f1s]
    vals_sorted  = [x[1] for x in f1s]
    bar_c = [gold_col if '⚠' not in n else '#ffbb78' for n in names_sorted]
    ax3.barh(names_sorted, vals_sorted, color=bar_c,
             edgecolor='black', linewidth=0.5, alpha=0.9)
    for i, v in enumerate(vals_sorted):
        ax3.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
    ax3.set_xlabel('Macro F1', fontsize=10)
    ax3.set_title(f'Vs {gold_name}', fontsize=11)
    ax3.set_xlim(0, 1.0)
    ax3.axvline(x=0.5, color='red', linestyle='--', alpha=0.3, linewidth=0.8)
    ax3.grid(axis='x', alpha=0.3)

fig3.suptitle('System Rankings Under Each Gold Standard
'
              '(Does pipeline advantage change under LLM gold?)',
              fontsize=12)
plt.tight_layout()
for ext in ['pdf', 'png']:
    fig3.savefig(FIGURES_DIR / f'nb14_rankings_dual_gold.{ext}',
                 dpi=300 if ext == 'pdf' else 150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: nb14_rankings_dual_gold.pdf/.png")


In [ ]:
# ── CELL 10 : SAVE SUMMARY JSON ───────────────────────────────────────────────

summary = {
    'notebook'            : '14_dual_gold_standard',
    'generated_at'        : datetime.now().isoformat(),
    'n_articles'          : len(SHARED_IDS),
    'article_set_note'    : ('Intersection of all four LLM DE runs. '
                             'Llama 70B DE is binding constraint (~162 articles). '
                             'All evaluations use identical article set for comparability.'),
    'gold_standards'      : {
        'pipeline_gold_de': {
            'description': 'Full spaCy + Stanza + Flair agreement',
            'n_entities'  : sum(len(pipeline_gold[a]) for a in SHARED_IDS),
            'mean_per_art': round(sum(len(pipeline_gold[a])
                                  for a in SHARED_IDS) / len(SHARED_IDS), 1),
            'label_dist'  : dict(pg_labels),
        },
        'llm_gold_de': {
            'description': 'Full Scout 17B + Qwen3 32B + Llama 70B DE agreement',
            'n_entities'  : sum(len(llm_gold_de[a]) for a in SHARED_IDS),
            'mean_per_art': round(sum(len(llm_gold_de[a])
                                  for a in SHARED_IDS) / len(SHARED_IDS), 1),
            'label_dist'  : dict(lg_labels),
        },
        'llm_gold_en': {
            'description': 'Full Scout 17B + Qwen3 32B + Llama 70B EN agreement',
            'n_entities'  : sum(len(llm_gold_en[a]) for a in SHARED_IDS),
            'mean_per_art': round(sum(len(llm_gold_en[a])
                                  for a in SHARED_IDS) / len(SHARED_IDS), 1),
            'label_dist'  : dict(le_labels),
        },
    },
    'fleiss_kappa_llm_de' : kappa_llm_de,
    'fleiss_kappa_pipeline': 0.717,
    'gold_overlap_jaccard' : round(float(np.mean(jaccard_scores)), 4),
    'f1_results'           : {
        sys_name: {
            gold_name: {'P': v['P'], 'R': v['R'], 'F1': v['F1'],
                        'per_label': v['per_label']}
            for gold_name, v in gold_results.items()
        }
        for sys_name, gold_results in results.items()
    },
    'notes': (
        'Pipeline systems evaluated on full article text (no truncation). '
        'LLM systems evaluated on 8000-char text limit (96.6% entity coverage). '
        'LLM EN-input comparisons vs German gold standards are language-gap comparisons '
        '(lower bounds). LLM EN-input vs LLM Gold EN is the clean English comparison. '
        'Fleiss kappa computed using entity presence/absence binary rating scheme.'
    ),
}

with open(DATA_PROC / 'nb14_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# ── Final print ───────────────────────────────────────────────────────────────
print("=" * 70)
print("NOTEBOOK 14 COMPLETE")
print("=" * 70)
print(f"  Articles          : {len(SHARED_IDS)}")
print(f"  Fleiss κ (LLM DE) : {kappa_llm_de}  "
      f"(pipeline κ=0.717 for reference)")
print(f"  Gold overlap      : {round(float(np.mean(jaccard_scores)),3)} "
      f"mean Jaccard (Pipeline Gold vs LLM Gold DE)")
print()
print("  Key F1 comparison (Pipeline Gold vs LLM Gold DE):")
print(f"  {'System':<45} {'Vs PG':>8} {'Vs LG':>8} {'Diff':>8}")
print(f"  {'-'*70}")
for sys_name in system_lookups:
    pg = results[sys_name]['Pipeline Gold (DE)']['F1']
    lg = results[sys_name]['LLM Gold DE']['F1']
    d  = lg - pg
    note = ' ⚠' if '(LLM EN)' in sys_name else ''
    print(f"  {sys_name+note:<45} {pg:>8.3f} {lg:>8.3f} {d:>+8.3f}")
print()
print("  Outputs:")
print("  nb14_f1_dual_gold.pdf/.png       — main F1 comparison figure")
print("  nb14_gold_comparison.pdf/.png    — gold standard size and label dist")
print("  nb14_rankings_dual_gold.pdf/.png — system rankings under each gold")
print("  nb14_summary.json                — full results")
